In [2]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [5]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if tool_choice:
        params["tool_choice"] = tool_choice

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


def run_batch(invocations=[]):
    batch_ouputs = []

    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        tool_ouptut = run_tool(name, args)
        batch_ouptuts.append({
            "tool_name": name,
            "output": tool_output
        })
    return batch_ouputs
    

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
        

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            batch_tool_schema
        ])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages


In [6]:
messages = []

add_user_message(
    messages,
    thinking_test_str
)

chat(messages, thinking=True)

Message(id='msg_01C6F2taP3DYkRJVCH9rPv9k', content=[ThinkingBlock(signature='EuIGCkYICxgCKkByOnjLM7YZh4ouKYYTdD3+LRzvd+mXKqmojDVYaUdJD+MIzCvgKX2wK8/IQssLkklJzWE/bTUPss8fz9hmBudjEgx+4hwlCs04Jn1I82EaDAwjLn+WnpBH0XuLVSIwsl7arnwoENRXG2xKhbXLksDKj0bL/TRsiEHvg+38FHzEDpe+BtWK1m0Ke2u7ILAPKskF6usbGriiLmqn/eR5E1bZVLObAxb0nlPNmUU7DcLXLLSSiN9dzT96P0AsaZ44pCXtzR+lQAjErm0/5CFn5ush532kVXblLo3qm+BWGSUpGH4GgkpQaFMJc5M9iqXVZRikpE9xmBodcJHFGUC2HiAxd6mEOA0UxVl+MsmXLg8KoWpV2HaHAVh8IuoD0HMa3xo88LmMNLZO+K8zBFMc7nrH58dxmUG+VtyNKeehBoxLaB8d6wTWW3imMxX6HBL7RuhCWDkiw55whuStcy8MJs7NFuJFNv5+oB8xoMRyihr5A+SWG7gvoLr6TIOWVEoJKmt1/mKVEr1rxQiJnA1oGwzko1Yn6USKcSAWyib19Gn3JsbVTZ9F7OdPKznNrvmuRtVIjj3ll9vEXG8cbBsFk1rZEubfSrC0K/hRZTtwyDHM41bEPexjve6fmUbckH6IUBfhBPH74C+U8XOfEY6ZReMZ8ZmhRYv3hNEBYMbQjkG16qwvGYRu3ZRatTo+SKZgkk9oiuVzDFpkT9xel+EKvaQkIek4usUJTl+uFGQSpj9guybtWg1CXQs0PXJLYYFYMiFBNzEusKCQoq9orUEGBgX4+rr+7VugF05cZN9KgtI1SnJhxhSNBjqiWzEyg4ytyRtTRhCz1xOG0wLNCNKhJX6jtgdsvvewn7BaME6ttNczU9v92eoT4D4UBNJtA0gBMlP/Rv7U5PM6Krt